# Challenge 2: Optimization of Battery Usage in the Installation

In this notebook, we address Objective 2 of the Repsol IE Sustainability Challenge:

- **Objective:** Optimize the use of a theoretical battery (100 kWh capacity, 100 kW max charge/discharge, one charge/discharge cycle per day) to maximize self‑consumption of solar energy and reduce grid dependence.

We will:

1. Load and prepare data (solar generation predictions, actual photovoltaic consumption, and grid consumption).
2. Ensure proper datetime handling (including timezone conversion) and filter for September 2024.
3. Merge datasets and compute the surplus solar energy.
4. Simulate battery operation using an improved strategy.
5. Compute the Self‑Consumption Ratio (Ra) as a business metric.
6. Export the final simulation results to CSV for submission.



In [ ]:
#import needed libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pytz

print('Libraries imported successfully!')

Libraries imported successfully!


## 1) Data Loading & Preparation

We load three datasets:

- **Solar Generation Predictions:** (from `september_predictions3.csv`)
- **Actual Photovoltaic Consumption:** (from `Consumo_Fotovoltaica.csv`; date column is "FECHA")
- **Grid Consumption:** (from `Consumo.csv`; date column is "FECHA")

We convert the datetime columns to the local time zone (`Europe/Madrid`) and filter for September 2024.

In [2]:
# Load solar generation predictions; assume the column is already 'datetime'
df_gen = pd.read_csv('september_predictionsv2.csv', parse_dates=['datetime'])

# Load actual photovoltaic consumption; the date column is 'FECHA'
df_consumption = pd.read_csv('Repsol Datasets/Consumo_Fotovoltaica.csv', parse_dates=['FECHA'])
df_consumption.rename(columns={'FECHA': 'datetime'}, inplace=True)

# Load grid consumption data; the date column is 'FECHA'
df_grid = pd.read_csv('Repsol Datasets/Consumo.csv', parse_dates=['FECHA'])
df_grid.rename(columns={'FECHA': 'datetime'}, inplace=True)

# Helper function to ensure datetime is in the desired timezone
def ensure_local_timezone(df, col, tz='Europe/Madrid'):
    if df[col].dt.tz is None:
        df[col] = df[col].dt.tz_localize('UTC')
    df[col] = df[col].dt.tz_convert(tz)
    return df

local_tz = 'Europe/Madrid'
df_gen = ensure_local_timezone(df_gen, 'datetime', local_tz)
df_consumption = ensure_local_timezone(df_consumption, 'datetime', local_tz)
df_grid = ensure_local_timezone(df_grid, 'datetime', local_tz)

# Filter for September 2024
sep_start = pd.Timestamp('2024-09-01 00:00:00', tz=local_tz)
sep_end   = pd.Timestamp('2024-09-30 23:00:00', tz=local_tz)

df_sep_gen = df_gen[(df_gen['datetime'] >= sep_start) & (df_gen['datetime'] <= sep_end)].copy()
df_sep_cons = df_consumption[(df_consumption['datetime'] >= sep_start) & (df_consumption['datetime'] <= sep_end)].copy()
df_sep_grid = df_grid[(df_grid['datetime'] >= sep_start) & (df_grid['datetime'] <= sep_end)].copy()

print('Solar generation predictions shape:', df_sep_gen.shape)
print('Photovoltaic consumption shape:', df_sep_cons.shape)
print('Grid consumption shape:', df_sep_grid.shape)
print('Expected rows:', 30 * 24)

Solar generation predictions shape: (718, 2)
Photovoltaic consumption shape: (720, 2)
Grid consumption shape: (2, 2)
Expected rows: 720


## 2) Merge & Compute Surplus Energy

We merge the solar generation predictions with the actual consumption data on `datetime` and calculate the surplus energy as:

```
Surplus = Predicted Solar Generation - Actual Photovoltaic Consumption
```

Negative values are clipped to 0.

In [3]:
# Merge solar generation predictions with consumption
# Note: We assume df_sep_gen has a column 'pv_generation_pred' for predictions
df_sep = pd.merge(df_sep_gen[['datetime', 'pv_generation_pred']], 
                  df_sep_cons[['datetime', 'TOTAL_KWH_ENERGIA']], 
                  on='datetime', 
                  how='left')

# Calculate surplus energy
df_sep['excess_energy'] = df_sep['pv_generation_pred'] - df_sep['TOTAL_KWH_ENERGIA']
df_sep['excess_energy'] = df_sep['excess_energy'].clip(lower=0)

print('Surplus energy calculated. Sample:')
display(df_sep.head(5))

Surplus energy calculated. Sample:


,datetime,pv_generation_pred,TOTAL_KWH_ENERGIA,excess_energy
0,2024-09-01 02:00:00+02:00,0.355018,0.0,0.355018
1,2024-09-01 03:00:00+02:00,-0.000158,0.0,0.000000
2,2024-09-01 04:00:00+02:00,-0.000004,0.0,0.000000
3,2024-09-01 05:00:00+02:00,-0.000004,0.0,0.000000
4,2024-09-01 06:00:00+02:00,-0.000004,0.0,0.000000


## 3) Improved Battery Simulation

We simulate a theoretical battery for each day in September with these rules:

- **Capacity:** 100 kWh
- **Max Charge/Discharge Power:** 100 kW per hour
- **Charging:** Battery charges with available surplus energy until full
- **Discharging:** Battery discharges at the hour with the highest grid consumption

This simulation will produce new columns:

- `battery_charge`: Battery level after each hour
- `energy_charged`: Energy charged during each hour
- `energy_discharged`: Energy discharged at the chosen hour

We then add a column `energy_recovered` (energy discharged) for business metric calculations.

In [ ]:
def simulate_battery(df_day, capacity=100, max_power=100):
    """
    Simulate battery operation for one day.
    
    Parameters:
        df_day: DataFrame for one day with 'excess_energy' and 'TOTAL_KWH_ENERGIA'.
        capacity: Battery capacity in kWh.
        max_power: Maximum charge/discharge in kWh per hour.
    
    Returns:
        DataFrame with additional columns:
            - 'battery_charge'
            - 'energy_charged'
            - 'energy_discharged'
    """
    battery_level = 0
    charge_list = []
    energy_charged = []
    energy_discharged = np.zeros(len(df_day))
    
    # Charge battery based on available surplus each hour
    for i, row in df_day.iterrows():
        available_surplus = row['excess_energy']
        charge = min(available_surplus, max_power, capacity - battery_level)
        battery_level += charge
        charge_list.append(battery_level)
        energy_charged.append(charge)
    
    df_day = df_day.copy()
    df_day['battery_charge'] = charge_list
    df_day['energy_charged'] = energy_charged
    
    # Determine discharge hour: choose hour with highest grid consumption
    if 'TOTAL_KWH_ENERGIA' in df_day.columns:
        discharge_idx = df_day['TOTAL_KWH_ENERGIA'].idxmax()
    else:
        discharge_idx = df_day.index[-1]
    
    # Discharge all energy (subject to max power constraint) at that hour
    discharge_amount = min(battery_level, max_power)
    energy_discharged[df_day.index.get_loc(discharge_idx)] = discharge_amount
    battery_level -= discharge_amount
    
    df_day['energy_discharged'] = energy_discharged
    df_day['energy_recovered'] = df_day['energy_discharged']
    
    return df_day

# Add a 'date' column and apply simulation for each day
df_sep['date'] = df_sep['datetime'].dt.date
df_simulated = df_sep.groupby('date').apply(lambda d: simulate_battery(d.copy()))

#display results
print('Battery simulation completed.')
display(df_simulated.head(10))

Battery simulation completed.


/var/folders/8m/_6ttb91j0dl6ptlnvls6lf040000gn/T/ipykernel_45860/3526582051.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_simulated = df_sep.groupby('date').apply(lambda d: simulate_battery(d.copy()))


datetime  pv_generation_pred  TOTAL_KWH_ENERGIA  \
date                                                                            
2024-09-01 0 2024-09-01 02:00:00+02:00            0.355018               0.00   
           1 2024-09-01 03:00:00+02:00           -0.000158               0.00   
           2 2024-09-01 04:00:00+02:00           -0.000004               0.00   
           3 2024-09-01 05:00:00+02:00           -0.000004               0.00   
           4 2024-09-01 06:00:00+02:00           -0.000004               0.00   
           5 2024-09-01 07:00:00+02:00           -0.000004               0.00   
           6 2024-09-01 08:00:00+02:00            0.004976               0.00   
           7 2024-09-01 09:00:00+02:00            0.183967               6.05   
           8 2024-09-01 10:00:00+02:00            0.524655              20.56   
           9 2024-09-01 11:00:00+02:00            5.741287              26.05   

              excess_energy        date  battery_charge  energy_charged  \
date                                                                      
2024-09-01 0       0.355018  2024-09-01        0.355018        0.355018   
           1       0.000000  2024-09-01        0.355018        0.000000   
           2       0.000000  2024-09-01        0.355018        0.000000   
           3       0.000000  2024-09-01        0.355018        0.000000   
           4       0.000000  2024-09-01        0.355018        0.000000   
           5       0.000000  2024-09-01        0.355018        0.000000   
           6       0.004976  2024-09-01        0.359994        0.004976   
           7       0.000000  2024-09-01        0.359994        0.000000   
           8       0.000000  2024-09-01        0.359994        0.000000   
           9       0.000000  2024-09-01        0.359994        0.000000   

              energy_discharged  energy_recovered  
date                                               
2024-09-01 0                0.0               0.0  
           1                0.0               0.0  
           2                0.0               0.0  
           3                0.0               0.0  
           4                0.0               0.0  
           5                0.0               0.0  
           6                0.0               0.0  
           7                0.0               0.0  
           8                0.0               0.0  
           9                0.0               0.0

## 4) Compute Business Metrics

### Self‑Consumption Ratio (Ra)

We define Ra as:

```
Ra = (Direct Solar Consumption + Energy Recovered from Battery) / Total Solar Generation
```

Here, we assume direct consumption is given by `TOTAL_KWH_ENERGIA` and energy recovered by battery is `energy_discharged`.

We calculate Ra for September 2024.

In [5]:
# Compute solar used: actual consumption plus energy discharged from battery
df_simulated['solar_used'] = df_simulated['TOTAL_KWH_ENERGIA'] + df_simulated['energy_discharged']

total_solar_gen = df_simulated['pv_generation_pred'].sum()
total_solar_used = df_simulated['solar_used'].sum()

Ra = total_solar_used / total_solar_gen
print(f"Self-Consumption Ratio (Ra): {Ra:.4f}")

Self-Consumption Ratio (Ra): 1.1533


## 5) Export Final Predictions

We ensure that our final simulation for September has exactly 720 rows and is sorted by datetime.
We then export the following columns to CSV:

- datetime
- pv_generation_pred
- TOTAL_KWH_ENERGIA
- energy_discharged
- solar_used

In [6]:
df_simulated

datetime  pv_generation_pred  \
date                                                           
2024-09-01 0   2024-09-01 02:00:00+02:00            0.355018   
           1   2024-09-01 03:00:00+02:00           -0.000158   
           2   2024-09-01 04:00:00+02:00           -0.000004   
           3   2024-09-01 05:00:00+02:00           -0.000004   
           4   2024-09-01 06:00:00+02:00           -0.000004   
...                                  ...                 ...   
2024-09-30 713 2024-09-30 19:00:00+02:00           63.435281   
           714 2024-09-30 20:00:00+02:00           51.041088   
           715 2024-09-30 21:00:00+02:00           23.764333   
           716 2024-09-30 22:00:00+02:00            5.302000   
           717 2024-09-30 23:00:00+02:00           -0.043139   

                TOTAL_KWH_ENERGIA  excess_energy        date  battery_charge  \
date                                                                           
2024-09-01 0                 0.00       0.355018  2024-09-01        0.355018   
           1                 0.00       0.000000  2024-09-01        0.355018   
           2                 0.00       0.000000  2024-09-01        0.355018   
           3                 0.00       0.000000  2024-09-01        0.355018   
           4                 0.00       0.000000  2024-09-01        0.355018   
...                           ...            ...         ...             ...   
2024-09-30 713              36.74      26.695281  2024-09-30       40.974349   
           714               4.98      46.061088  2024-09-30       87.035437   
           715              -0.04      23.804333  2024-09-30      100.000000   
           716               0.00       5.302000  2024-09-30      100.000000   
           717               0.00       0.000000  2024-09-30      100.000000   

                energy_charged  energy_discharged  energy_recovered  \
date                                                                  
2024-09-01 0          0.355018                0.0               0.0   
           1          0.000000                0.0               0.0   
           2          0.000000                0.0               0.0   
           3          0.000000                0.0               0.0   
           4          0.000000                0.0               0.0   
...                        ...                ...               ...   
2024-09-30 713       26.695281                0.0               0.0   
           714       46.061088                0.0               0.0   
           715       12.964563                0.0               0.0   
           716        0.000000                0.0               0.0   
           717        0.000000                0.0               0.0   

                solar_used  
date                        
2024-09-01 0          0.00  
           1          0.00  
           2          0.00  
           3          0.00  
           4          0.00  
...                    ...  
2024-09-30 713       36.74  
           714        4.98  
           715       -0.04  
           716        0.00  
           717        0.00  

[718 rows x 10 columns]

In [7]:
# Sort the simulated data by datetime
df_simulated.sort_values('datetime', inplace=True)

# Verify row count
print('Number of rows in simulated September data:', df_simulated.shape[0])

# Export final predictions to CSV
export_cols = ['datetime', 'pv_generation_pred', 'TOTAL_KWH_ENERGIA', 'energy_discharged', 'solar_used']
df_simulated[export_cols].to_csv('Predictions/Battery_Predictions.csv', index=False)
print("Predictions exported to 'Predictions/Battery_Predictions.csv'")

Number of rows in simulated September data: 718
Predictions exported to 'Predictions/Battery_Predictions.csv'


## Wrap-Up & Conclusion

In this notebook we:

1. Loaded solar generation predictions, photovoltaic consumption, and grid consumption data.
2. Converted datetime columns to local time and filtered for September 2024 (ensuring 720 rows).
3. Merged the datasets and calculated surplus solar energy.
4. Simulated battery operation with an improved strategy (charging with surplus and discharging at peak grid consumption).
5. Computed the Self‑Consumption Ratio (Ra) as a key business metric.
6. Exported the final simulation results for further evaluation.

Further refinements could include more dynamic battery simulation, incorporation of carbon intensity data, and advanced feature engineering. Iterative improvements will help achieve a lower MAE and better business metrics.

